# Census Block Group Data
Pulls **population** and **family** estimates from the ACS 5-Year estimates via the Census Bureau API.

**A free Census API key is required.** Sign up at https://api.census.gov/data/key_signup.html — the key arrives by email within a few minutes. Paste it into `CENSUS_API_KEY` in the configuration cell below.

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────

# Add or remove block group GEOIDs here (11 or 12 digits both accepted)
BLOCK_GROUP_GEOIDS = [
    "60750252002",
    "60750253004",
    "60750251001",
    "60750251002",
]

ACS_YEAR = 2022          # most recent available ACS 5-year release

# Set your key via environment variable: export CENSUS_API_KEY="your-key"
# Sign up free at https://api.census.gov/data/key_signup.html
import os
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "")

In [9]:
import requests
import pandas as pd
from collections import defaultdict

if not CENSUS_API_KEY:
    raise ValueError(
        "CENSUS_API_KEY is required for ACS block group queries.\n"
        "Sign up free at https://api.census.gov/data/key_signup.html"
    )

# ACS variables to fetch
# Reference: https://api.census.gov/data/{year}/acs/acs5/variables.html
VARIABLES = {
    "B01003_001E": "Total Population",
    "B11001_001E": "Total Households",
    "B11001_002E": "Family Households",
    "B11001_003E": "Married-Couple Families",
    "B11001_005E": "Male Householder, No Spouse",
    "B11001_006E": "Female Householder, No Spouse",
    "B11001_007E": "Nonfamily Households",
}

BASE_URL = f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5"

In [10]:
# ── KEY DIAGNOSTIC ───────────────────────────────────────────────────────────
# Run this cell to verify your key before making data requests.
import requests

key = CENSUS_API_KEY.strip()
print(f"Key length : {len(key)}")
print(f"First/last : '{key[:4]}...{key[-4:]}'")

test = requests.get(
    "https://api.census.gov/data/2022/acs/acs5",
    params={"get": "NAME,B01003_001E", "for": "state:06", "key": key},
    timeout=15,
)
print(f"Status     : {test.status_code}")
if test.ok:
    print("Key is valid ✓")
else:
    print(f"Response   : {test.text[:300]}")

Key length : 40
First/last : 'df65...2ded'
Status     : 200
Key is valid ✓


In [11]:
def parse_geoid(geoid: str) -> dict:
    """Parse a 11- or 12-digit block group GEOID into its FIPS components."""
    geoid = geoid.strip().zfill(12)  # pad to 12 digits
    return {
        "geoid": geoid,
        "state": geoid[0:2],
        "county": geoid[2:5],
        "tract": geoid[5:11],
        "block_group": geoid[11:12],
    }


def fetch_block_groups(state: str, county: str, tract: str) -> list[dict]:
    """Fetch all block groups in a given tract from the Census API."""
    params = {
        "get": ",".join(["NAME"] + list(VARIABLES.keys())),
        "for": "block group:*",
        "in": f"state:{state} county:{county} tract:{tract}",
    }
    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY

    resp = requests.get(BASE_URL, params=params, timeout=30)
    if not resp.ok:
        raise RuntimeError(f"Census API {resp.status_code}: {resp.text[:500]}")
    try:
        rows = resp.json()
    except Exception:
        raise RuntimeError(f"Non-JSON response from Census API:\n{resp.text[:1000]}")
    headers, *data = rows
    return [dict(zip(headers, row)) for row in data]


# Deduplicate and parse input GEOIDs
parsed = {g: parse_geoid(g) for g in dict.fromkeys(BLOCK_GROUP_GEOIDS)}

# Group by (state, county, tract) to minimize API calls
tract_groups: dict[tuple, list] = defaultdict(list)
for p in parsed.values():
    tract_groups[(p["state"], p["county"], p["tract"])].append(p["geoid"])

print(f"Fetching data for {len(parsed)} unique block groups across {len(tract_groups)} tract(s)...")

# Debug: show what will be requested
for (state, county, tract) in tract_groups:
    url = f"{BASE_URL}?get=NAME,B01003_001E&for=block+group:*&in=state:{state}+county:{county}+tract:{tract}"
    print(f"  → state={state} county={county} tract={tract}")

Fetching data for 6 unique block groups across 3 tract(s)...
  → state=06 county=075 tract=025200
  → state=06 county=075 tract=025300
  → state=06 county=075 tract=025100


In [12]:
all_rows = []

for (state, county, tract), target_geoids in tract_groups.items():
    records = fetch_block_groups(state, county, tract)
    for rec in records:
        full_geoid = state + county + rec["tract"] + rec["block group"]
        if full_geoid in target_geoids:
            row = {"GEOID": full_geoid, "Name": rec["NAME"]}
            for var_code, label in VARIABLES.items():
                row[label] = int(rec[var_code]) if rec[var_code] not in (None, "-666666666") else None
            all_rows.append(row)

df = pd.DataFrame(all_rows).set_index("GEOID")

# Preserve input order
ordered_geoids = [parse_geoid(g)["geoid"] for g in dict.fromkeys(BLOCK_GROUP_GEOIDS)]
df = df.reindex(ordered_geoids)

print(f"Retrieved {len(df)} block groups  (ACS 5-Year, {ACS_YEAR})")
df

Retrieved 6 block groups  (ACS 5-Year, 2022)


,Name,Total Population,Total Households,Family Households,Married-Couple Families,"Male Householder, No Spouse","Female Householder, No Spouse",Nonfamily Households
GEOID,,,,,,,,
060750252002,Block Group 2; Census Tract 252; San Francisco...,1574,647,322,290,32,0,325
060750252001,Block Group 1; Census Tract 252; San Francisco...,1528,605,369,280,0,89,236
060750253004,Block Group 4; Census Tract 253; San Francisco...,2073,726,358,253,78,27,368
060750251001,Block Group 1; Census Tract 251; San Francisco...,1919,741,445,248,81,116,296
060750251002,Block Group 2; Census Tract 251; San Francisco...,839,321,226,183,10,33,95
060750253001,Block Group 1; Census Tract 253; San Francisco...,756,332,195,141,7,47,137


In [13]:
# ── MAP ──────────────────────────────────────────────────────────────────────
# Requires: pip install pygris geopandas folium
try:
    import folium
    import geopandas as gpd
    from pygris import block_groups as fetch_bg_shapes
    from shapely.geometry import mapping
except ImportError as e:
    raise ImportError(
        f"Missing dependency: {e.name}\n"
        "Install with:  pip install pygris geopandas folium"
    )

target_geoids = [parse_geoid(g)["geoid"] for g in dict.fromkeys(BLOCK_GROUP_GEOIDS)]

# Download SF county block group boundaries (cached locally after first run)
gdf = fetch_bg_shapes(state="06", county="075", year=ACS_YEAR, cache=True)
gdf = gdf[gdf["GEOID"].isin(target_geoids)]
gdf = gdf[["GEOID", "geometry"]].merge(df.reset_index(), on="GEOID", how="left")

# Build GeoJSON manually to avoid a NumPy 2.x bug in geopandas .to_json()
features = []
for _, row in gdf.iterrows():
    features.append({
        "type": "Feature",
        "geometry": mapping(row.geometry),
        "properties": {
            "GEOID": row["GEOID"],
            "Total Population": row["Total Population"],
            "Family Households": row["Family Households"],
        },
    })
geojson_data = {"type": "FeatureCollection", "features": features}

# Center map on bounding box midpoint
minx, miny, maxx, maxy = gdf.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

m = folium.Map(location=center, zoom_start=14, tiles="CartoDB positron")

folium.GeoJson(
    geojson_data,
    style_function=lambda _: {
        "fillColor": "#2563eb",
        "color": "#1d4ed8",
        "weight": 2,
        "fillOpacity": 0.4,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["GEOID", "Total Population", "Family Households"],
        aliases=["GEOID", "Population", "Family Households"],
    ),
).add_to(m)

m

In [14]:
# Summary totals across all selected block groups
numeric_cols = [c for c in df.columns if c != "Name"]
totals = df[numeric_cols].sum().rename("TOTAL")
pd.DataFrame(totals).T

,Total Population,Total Households,Family Households,Married-Couple Families,"Male Householder, No Spouse","Female Householder, No Spouse",Nonfamily Households
TOTAL,8689,3372,1915,1395,208,312,1457
